In [ ]:
%pip install openai
%pip install pydantic-settings
%pip install tiktoken
%pip install requests

In [ ]:
from pydantic_settings import BaseSettings


class Env(BaseSettings):
    AZURE_OPENAI_ENDPOINT: str
    AZURE_OPENAI_API_KEY: str
    AZURE_OPENAI_API_VERSION: str = "2023-12-01-preview"
    GPT_35_MODEL_NAME: str = "gpt-35-turbo"
    GPT_4_MODEL_NAME: str = "gpt-4"
    TEXT_EMBEDDING_MODEL_NAME: str = "text-embedding-ada-002"

    class Config:
        env_file = 'function_calling.env'


env = Env()

## Function Calling
- https://learn.microsoft.com/zh-tw/azure/ai-services/openai/how-to/function-calling?tabs=python
- https://platform.openai.com/docs/guides/function-calling/function-calling

In [ ]:
from openai import AzureOpenAI

openai = AzureOpenAI(azure_endpoint=env.AZURE_OPENAI_ENDPOINT, api_key=env.AZURE_OPENAI_API_KEY,
                     api_version=env.AZURE_OPENAI_API_VERSION)

In [ ]:
get_random_user_tool = {
    "type": "function",
    "function": {
        "name": "get_random_user",
        "description": "Want to create random fake user for your application",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    }
}

get_taipei_library_seats_tool = {
    "type": "function",
    "function": {
        "name": "get_taipei_library_seats",
        "description": "查詢台北市圖書館的空位",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    }
}

In [ ]:
import requests

def get_random_user():
    return requests.get('https://randomuser.me/api/').json()

def get_taipei_library_seats():
    return requests.get("https://seat.tpml.edu.tw/sm/service/getAllArea").json()

available_functions = {
    "get_random_user": get_random_user,
    "get_taipei_library_seats": get_taipei_library_seats
}

In [ ]:
from datetime import datetime
import json

sys_prompt = f"Today is {datetime.now()}."

questions = [
    '想生資料',
    '我在台北走得好累，好想找圖書館休息，哪邊可以休息'
]

for question in questions:
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": question}
    ]

    response = openai.chat.completions.create(
        model=env.GPT_35_MODEL_NAME,
        messages=messages,
        tools=[get_random_user_tool, get_taipei_library_seats_tool],
        tool_choice="auto",  # auto is default, but we'll be explicit
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        messages.append(response_message)
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_call()
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": str(function_response),
                }
            )  # extend conversation with function response

        second_response = openai.chat.completions.create(
            model=env.GPT_4_MODEL_NAME,
            messages=messages,
        )
        print(f"Second Response: {second_response.choices[0].message.content}")
    else:
        print(f"First Response: {response_message.content}")